# Approach 1 (avoid-step, elbow distance method) — Averaged CE vs Fitted Curve

For every (P%, BS) combination: scatter shows averaged raw CE **up to the detected step cutoff**,
overlaid with the smooth fitted curve `A + B/(BN+1)^n` where **A is the empirical tail floor**.

The elbow is found via the **perpendicular distance method on the actual data**:
1. Draw a reference line from the **start point** `(BN[0], CE[0])` to the **end point** `(BN[-1], A)`
2. Normalize both axes to [0, 1] so BN and CE scales don't interfere
3. In normalized space the line runs from `(0, 1)` to `(1, 0)` — equation `x + y = 1`
4. Elbow = data point with maximum perpendicular distance from this line ∝ `x_norm + y_norm − 1`

Each plot has a **distance-method inset** (bottom-right) showing the normalized data cloud, the
reference diagonal, the elbow point, and the perpendicular drop — making the detection transparent.

Parameters come from `approach_1_elbow_step/intermediate/approach_1_fit_params_bs_{bs}.csv`.

PNGs saved to `BS_{bs}/fitting_avg_plot_A_1_elbow_step_p_{p}_bs_{bs}.png`.

In [ ]:
# === Cell 1 — Config, imports ===
import os, glob, re
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

# ── CONFIG ────────────────────────────────────────────────────────────────────────────
BN_STEP_MIN = 100    # informational only
STEP_THRESH = 0.01
# ───────────────────────────────────────────────────────────────────────────────

BASE_DIR  = r"C:\Users\Student\Desktop\Projects\research\physlab\SLP\SLP-MNIST\prune_layers_ALL"
INTER_DIR = r"C:\Users\Student\Desktop\Projects\research\physlab\SLP\SLP-MNIST\IPA_methods\Approach_1\test_1\approach_1_elbow_step\intermediate"
OUT_DIR   = r"C:\Users\Student\Desktop\Projects\research\physlab\SLP\SLP-MNIST\IPA_methods\Approach_1\test_1\approach_1_elbow_step\avg_plot_v_fitting_curve_elbow"

BATCH_SIZES = [64, 1024, 60000]
CE_o = np.log(10)   # ln(10) ≈ 2.302585

p_dirs = glob.glob(os.path.join(BASE_DIR, "p-percentage_*"))
PRUNING_LEVELS = sorted([
    float(re.search(r"p-percentage_([\d.]+)", d).group(1))
    for d in p_dirs
])
print(f"Found {len(PRUNING_LEVELS)} pruning levels: {PRUNING_LEVELS}")
print(f"CE_o = ln(10) = {CE_o:.6f}")
print("Cell 1 ready.")

In [ ]:
# === Cell 2 — Load all (P%, BS) records ===
records = []   # one dict per (p, bs)

for bs in BATCH_SIZES:
    params_csv = os.path.join(INTER_DIR, f"approach_1_fit_params_bs_{bs}.csv")
    if not os.path.exists(params_csv):
        print(f"[SKIP] Missing params CSV: {params_csv}")
        continue
    params_df = pd.read_csv(params_csv)
    params_df.columns = params_df.columns.str.strip()

    for p in PRUNING_LEVELS:
        avg_csv = os.path.join(BASE_DIR, f"p-percentage_{p}", f"batch_size_{bs}",
                               f"averaged_runs_p_{p}_bs_{bs}.csv")
        if not os.path.exists(avg_csv):
            print(f"  [SKIP] P%={p*100:5.1f}%  BS={bs:>6}  — no averaged CSV")
            continue

        row = params_df[np.isclose(params_df["P%"], p * 100)]
        if row.empty:
            print(f"  [SKIP] P%={p*100:5.1f}%  BS={bs:>6}  — no fit params row")
            continue

        cutoff_BN = float(row["cutoff_BN"].iloc[0])

        full_df = pd.read_csv(avg_csv)
        full_df.columns = full_df.columns.str.strip()
        ce_col = next((c for c in full_df.columns if c in ("Avg_CE_Test", "Avg_CE_test")), None)
        bn_col = next((c for c in full_df.columns if "Batch" in c), None)
        if ce_col is None or bn_col is None:
            print(f"  [SKIP] P%={p*100:5.1f}%  BS={bs:>6}  — unexpected columns {list(full_df.columns)}")
            continue
        full_df = full_df.dropna(subset=[ce_col, bn_col])
        max_bn  = float(full_df[bn_col].max())
        step_was_detected = cutoff_BN < max_bn

        avg_df = full_df[full_df[bn_col] < cutoff_BN]
        bn_avg = avg_df[bn_col].values.astype(float)
        ce_avg = avg_df[ce_col].values.astype(float)

        A          = float(row["A"].iloc[0])
        B          = float(row["B"].iloc[0])
        n          = float(row["n"].iloc[0])
        elbow_BN   = float(row["elbow_BN"].iloc[0])
        BN_learned = float(row["BN_learned"].iloc[0])
        CE_learned = float(row["CE_learned"].iloc[0])
        IPA        = float(row["IPA"].iloc[0])
        rmse_full  = float(row["RMSE_full"].iloc[0])   if "RMSE_full"   in row.columns else np.nan
        rmse_last50= float(row["RMSE_last50"].iloc[0]) if "RMSE_last50" in row.columns else np.nan

        from_data = np.isfinite(BN_learned) and (BN_learned <= max_bn)

        # Index of the elbow data point (used for the distance inset)
        if np.isfinite(elbow_BN) and len(bn_avg) > 0:
            elbow_idx = int(np.argmin(np.abs(bn_avg - elbow_BN)))
        else:
            elbow_idx = None

        records.append({
            "bs": bs, "p": p,
            "bn_avg":            bn_avg,
            "ce_avg":            ce_avg,
            "A":                 A,
            "B":                 B,
            "n":                 n,
            "elbow_BN":          elbow_BN,
            "elbow_idx":         elbow_idx,
            "BN_learned":        BN_learned,
            "CE_learned":        CE_learned,
            "from_data":         from_data,
            "IPA":               IPA,
            "cutoff_BN":         cutoff_BN,
            "step_was_detected": step_was_detected,
            "RMSE_full":         rmse_full,
            "RMSE_last50":       rmse_last50,
        })
        step_tag = "[step detected]" if step_was_detected else "[no step]"
        print(f"  OK   P%={p*100:5.1f}%  BS={bs:>6}  cutoff_BN={cutoff_BN:>6.0f} {step_tag:<16}  "
              f"A(floor)={A:.4f}  elbow_BN={elbow_BN:>7.1f}  BN_learned={BN_learned!r}")

print(f"\nLoaded {len(records)} combinations.")

In [ ]:
# === Cell 3 — Plot averaged CE (pre-step) vs fitted curve with elbow annotation ===
#
# Each plot shows:
#   - Grey scatter:          averaged CE data (pre-step window)
#   - Blue line:             fitted curve  A + B/(BN+1)^n  (A pinned to floor)
#   - Green dashed:          A = floor asymptote
#   - Grey dotted:           CE_o reference
#   - Purple star on scatter:elbow point (max perpendicular distance from reference line)
#   - Purple dashed horiz:   CE_learned  (CE at the elbow data point)
#   - Red vertical dashed:   BN_learned  (= elbow_BN, since elbow IS a data point)
#   - Bottom-right inset:    normalized data + reference diagonal + perpendicular drop

plt.rcParams.update({"font.size": 13})

for rec in records:
    bs                = rec["bs"]
    p                 = rec["p"]
    bn_avg            = rec["bn_avg"]
    ce_avg            = rec["ce_avg"]
    A                 = rec["A"]
    B                 = rec["B"]
    n                 = rec["n"]
    elbow_BN          = rec["elbow_BN"]
    elbow_idx         = rec["elbow_idx"]
    BN_learned        = rec["BN_learned"]
    CE_learned        = rec["CE_learned"]
    from_data         = rec["from_data"]
    IPA               = rec["IPA"]
    cutoff_BN         = rec["cutoff_BN"]
    step_was_detected = rec["step_was_detected"]
    RMSE_full         = rec["RMSE_full"]
    RMSE_last50       = rec["RMSE_last50"]

    bn_end    = max(bn_avg.max() if len(bn_avg) > 0 else cutoff_BN,
                    BN_learned if np.isfinite(BN_learned) else 0) * 1.3
    bn_smooth = np.linspace(0, bn_end, 800)
    y_fit     = A + B / ((bn_smooth + 1) ** n)

    fig, ax = plt.subplots(figsize=(10, 6))

    # 1. Averaged CE scatter (pre-step only)
    scatter_label = (r"$\overline{CE}_{test}$ (avg 100 runs, pre-step)"
                     if step_was_detected else
                     r"$\overline{CE}_{test}$ (avg 100 runs, full data)")
    ax.scatter(bn_avg, ce_avg, s=8, color="#aaaaaa", alpha=0.6, zorder=1,
               label=scatter_label)

    # 2. Smooth fitted curve
    ax.plot(bn_smooth, y_fit, color="#1f77b4", linewidth=2.2, zorder=3,
            label=(f"Fit (A=floor): A={A:.4f},  B={B:.4f},  n={n:.4f}\n"
                   f"RMSE_last50={RMSE_last50:.4f}  (RMSE_full={RMSE_full:.4f})"))

    # 3. Step cutoff boundary line
    if step_was_detected:
        ax.axvline(cutoff_BN, color="#bbbbbb", linewidth=1.0, linestyle="-", alpha=0.6)
        ax.text(cutoff_BN + 5, CE_o - 0.05,
                f"BN={cutoff_BN:.0f}\n(step cutoff)",
                fontsize=8, color="#888888", va="top")

    # 4. CE_o reference (dotted grey)
    ax.axhline(CE_o, color="#888888", linewidth=1.0, linestyle=":")
    ax.text(bn_end, CE_o + 0.03, f"CE_o = {CE_o:.4f}",
            ha="right", fontsize=10, color="#666666")

    # 5. A = floor asymptote (green dashed)
    ax.axhline(A, color="#2ca02c", linewidth=1.4, linestyle="--")
    ax.text(bn_end, A - 0.07, f"A = {A:.4f}  (floor asymptote)",
            ha="right", fontsize=10, color="#2ca02c")

    # 6. Elbow point on the scatter — the data point with max perpendicular distance
    if elbow_idx is not None and elbow_idx < len(bn_avg):
        ax.scatter([bn_avg[elbow_idx]], [ce_avg[elbow_idx]], s=160, color="#9467bd",
                   marker="*", zorder=6,
                   label=f"Elbow (max dist. from ref. line)  BN={elbow_BN:.1f}")

    # 7. CE_learned horizontal line (purple dashed)
    if np.isfinite(CE_learned):
        ax.axhline(CE_learned, color="#9467bd", linewidth=1.4, linestyle="--")
        ax.text(bn_end, CE_learned + 0.03, f"CE_learned = {CE_learned:.4f}",
                ha="right", fontsize=10, color="#9467bd")

    # 8. BN_learned vertical line + annotation
    if np.isfinite(BN_learned) and BN_learned > 0:
        bnl_color = "#d62728"
        ax.axvline(BN_learned, color=bnl_color, linewidth=1.4,
                   linestyle="--", alpha=0.8, zorder=4)
        annot_y = CE_learned if np.isfinite(CE_learned) else ce_avg.mean()
        ax.annotate(
            f"elbow_BN = {elbow_BN:.1f}\nBN_learned = {BN_learned:.0f}\n"
            f"CE_learned = {CE_learned:.4f}\nIPA = {IPA:.5f}",
            xy=(BN_learned, annot_y),
            xytext=(BN_learned + bn_end * 0.03, annot_y + 0.15),
            fontsize=9, color=bnl_color,
            arrowprops=dict(arrowstyle="->", color=bnl_color, lw=1.0)
        )

    ax.set_xlabel("Batch Number (BN)")
    ax.set_ylabel("CE_TEST")
    ax.set_xlim(0, bn_end)
    ax.set_ylim(max(0, A - 0.15), CE_o + 0.25)
    step_tag = f"step at BN={cutoff_BN:.0f}" if step_was_detected else "no step detected"
    ax.set_title(
        f"Approach 1 (avoid-step, elbow distance) — Avg CE vs Fit  |  P%={p*100:.1f}%  BS={bs}\n"
        f"Cutoff: {step_tag}   |   A(floor)={A:.4f}   RMSE_last50={RMSE_last50:.4f}"
    )
    ax.legend(fontsize=10, frameon=False, loc="upper right")
    ax.grid(True, alpha=0.25)

    # ── Distance-method inset ─────────────────────────────────────────────────────────
    # Shows the normalized (BN, CE) data, the reference diagonal from start→(BN_end,A),
    # the elbow point (★), and the perpendicular drop to the line.
    BN_range_n = bn_avg[-1] - bn_avg[0] if len(bn_avg) > 1 else 1.0
    CE_range_n = ce_avg[0] - A if len(ce_avg) > 0 and (ce_avg[0] - A) > 1e-10 else None

    if CE_range_n is not None and BN_range_n > 0:
        x_n = (bn_avg - bn_avg[0]) / BN_range_n
        y_n = (ce_avg - A)         / CE_range_n

        ax_ins = ax.inset_axes([0.57, 0.08, 0.40, 0.32])
        ax_ins.scatter(x_n, y_n, s=6, color="#aaaaaa", alpha=0.5, zorder=1)

        # Reference line: (0,1) → (1,0)
        ax_ins.plot([0, 1], [1, 0], color="#555555", linewidth=1.2,
                    linestyle="--", zorder=2, label="start → (BN_end, A)")

        # Start point marker
        ax_ins.scatter([0], [1], s=40, color="#2ca02c", marker="o", zorder=4)
        # End point marker (BN_end, A) → normalized to (1, 0)
        ax_ins.scatter([1], [0], s=40, color="#2ca02c", marker="s", zorder=4)

        # Elbow point and perpendicular drop
        if elbow_idx is not None and elbow_idx < len(x_n):
            xe, ye = x_n[elbow_idx], y_n[elbow_idx]
            ax_ins.scatter([xe], [ye], s=100, color="#9467bd", marker="*", zorder=5)
            # Foot of perpendicular from (xe, ye) to line x + y = 1:
            #   xf = (xe - ye + 1) / 2,  yf = 1 - xf
            xf = (xe - ye + 1) / 2
            yf = 1.0 - xf
            ax_ins.plot([xe, xf], [ye, yf], color="#9467bd",
                        linewidth=1.2, linestyle=":", zorder=3)

        ax_ins.set_xlabel("BN (norm.)", fontsize=8)
        ax_ins.set_ylabel("CE (norm.)", fontsize=8)
        ax_ins.set_title("Distance method", fontsize=8)
        ax_ins.tick_params(labelsize=7)
        ax_ins.set_xlim(-0.05, 1.08)
        ax_ins.set_ylim(-0.10, 1.12)
        ax_ins.grid(True, alpha=0.2)
        ax_ins.set_facecolor("#f9f9f9")
    # ─────────────────────────────────────────────────────────────────────────────────

    bs_dir  = os.path.join(OUT_DIR, f"BS_{bs}")
    os.makedirs(bs_dir, exist_ok=True)
    out_png = os.path.join(bs_dir, f"fitting_avg_plot_A_1_elbow_step_p_{p}_bs_{bs}.png")
    plt.tight_layout()
    plt.savefig(out_png, dpi=150, bbox_inches="tight")
    plt.close(fig)
    step_label = f"step@{cutoff_BN:.0f}" if step_was_detected else "no step"
    print(f"  Saved: fitting_avg_plot_A_1_elbow_step_p_{p}_bs_{bs}.png  "
          f"[{step_label}, elbow_BN={elbow_BN:.1f}, BN_learned={BN_learned}, RMSE_last50={RMSE_last50:.4f}]")

print("\n[Done]")